# 🤖 Machine Learning — Next Topics Notebook
## Random Forest → Feature Scaling → KNN → Cross Validation → Hyperparameter Tuning → K-Means

---

### Is Notebook Mein Kya Seekhoge?

| # | Topic | Type | Kya Karenge |
|---|-------|------|-------------|
| 1 | **Random Forest** | Supervised | Decision Tree ka powerful version |
| 2 | **Feature Scaling** | Preprocessing | Numbers ko same scale pe lao |
| 3 | **K-Nearest Neighbors** | Supervised | Neighbors dekh ke decide karo |
| 4 | **Cross Validation** | Evaluation | Model ko properly test karo |
| 5 | **Hyperparameter Tuning** | Optimization | Best settings auto dhundho |
| 6 | **K-Means Clustering** | Unsupervised | Khud groups banao (no target!) |

> **Ye notebook pehli wali ke baad hai** — Linear Regression, Logistic Regression, Decision Tree already seekh liye ho!


## ⚙️ Setup — Libraries Install & Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)

print('✅ Sab libraries load ho gayi!')

---
## 📊 Shared Dataset — Puri Notebook Mein Yahi Use Hoga

Student dataset — features se grade (A/B/C) predict karenge.


In [ ]:
np.random.seed(42)
n = 600

df = pd.DataFrame({
    'study_hours': np.random.uniform(1, 9, n).round(1),
    'attendance':  np.random.uniform(55, 100, n).round(1),
    'sleep_hours': np.random.uniform(4, 9, n).round(1),
    'tv_hours':    np.random.uniform(0, 6, n).round(1),
    'prev_score':  np.random.normal(62, 14, n).clip(25, 100).round(1),
})

df['marks'] = (
    df['study_hours'] * 8   +
    df['attendance']  * 0.3 +
    df['sleep_hours'] * 2   +
    df['tv_hours']    * (-5)+
    df['prev_score']  * 0.4 +
    np.random.normal(0, 7, n)
).clip(25, 100).round(1)

def grade(m):
    if m >= 75:   return 'A'
    elif m >= 55: return 'B'
    else:         return 'C'

df['grade'] = df['marks'].apply(grade)

print('Dataset shape:', df.shape)
print()
print('Grade distribution:')
print(df['grade'].value_counts())
print()
df.head()

In [ ]:
X = df[['study_hours', 'attendance', 'sleep_hours', 'tv_hours', 'prev_score']]
y = df['grade']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training: {len(X_train)} rows')
print(f'Testing:  {len(X_test)} rows')

---
---
# 🌲 PART 1 — RANDOM FOREST
---
---

## 📖 Theory — Random Forest Kya Hai?

### Simple Idea
> Akela Decision Tree overfit ho sakta hai. Agar **100 alag-alag trees** banao aur sab ka **vote lo** — toh answer zyada reliable hoga!

```
Random Forest = Baht Saare Decision Trees ka Ensemble

Tree 1 → "A"  ↘
Tree 2 → "B"   →  VOTE  →  "A" (majority) ✅
Tree 3 → "A"  ↗
Tree 4 → "A"
Tree 5 → "B"
```

### Kab Use Karein?
- Jab Decision Tree overfit ho raha ho
- Jab accuracy improve karni ho bina zyada kaam ke
- **Almost always Decision Tree se better!**


In [ ]:
# Random Forest — Train karo
rf_model = RandomForestClassifier(
    n_estimators=100,       # 100 trees banao
    max_depth=5,            # har tree ki max depth
    min_samples_split=10,   # node split ke liye min samples
    random_state=42,
    n_jobs=-1               # parallel processing — fast!
)

rf_model.fit(X_train, y_train)
print('✅ Random Forest train ho gaya!')
print(f'Trees: {rf_model.n_estimators}')

In [ ]:
# Predict aur Evaluate
y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f'Random Forest Accuracy: {acc_rf*100:.1f}%')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Decision Tree vs Random Forest comparison
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
acc_dt = accuracy_score(y_test, dt_model.predict(X_test))

dt_train = accuracy_score(y_train, dt_model.predict(X_train))
rf_train = accuracy_score(y_train, rf_model.predict(X_train))

print('=== COMPARISON ===')
print(f'Decision Tree → Train: {dt_train*100:.1f}%  Test: {acc_dt*100:.1f}%')
print(f'Random Forest → Train: {rf_train*100:.1f}%  Test: {acc_rf*100:.1f}%')
print()
print(f'Improvement: +{(acc_rf - acc_dt)*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = ['Decision Tree', 'Random Forest']
train_accs = [dt_train*100, rf_train*100]
test_accs  = [acc_dt*100, acc_rf*100]
x = np.arange(len(models))
w = 0.35

bars1 = axes[0].bar(x - w/2, train_accs, w, label='Training', color='steelblue')
bars2 = axes[0].bar(x + w/2, test_accs,  w, label='Testing',  color='tomato')
axes[0].set_xticks(x); axes[0].set_xticklabels(models)
axes[0].set_ylabel('Accuracy %'); axes[0].set_ylim(60, 105)
axes[0].set_title('Decision Tree vs Random Forest', fontweight='bold')
axes[0].legend()
for bar in bars1: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{bar.get_height():.1f}%', ha='center', fontsize=9)
for bar in bars2: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{bar.get_height():.1f}%', ha='center', fontsize=9)

importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()
axes[1].barh(importance.index, importance.values*100, color=sns.color_palette('Set2', 5))
axes[1].set_xlabel('Importance %')
axes[1].set_title('Random Forest — Feature Importance', fontweight='bold')

plt.tight_layout(); plt.show()

## ✅ Random Forest — Key Takeaways

| Cheez | Matlab |
|-------|--------|
| **n_estimators** | Kitne trees (100 default — zyada = better lekin slow) |
| **max_depth** | Har tree ki depth limit — overfitting control |
| **Bagging** | Har tree alag data subset pe train hoti hai |
| **feature_importances_** | Khud batata hai kaunsa feature important tha |
| **n_jobs=-1** | Saare CPU cores use karo |
| **Kab use karein** | Decision Tree se almost hamesha better! |


---
---
# ⚖️ PART 2 — FEATURE SCALING
---
---

## 📖 Theory — Scaling Kyun Zaroori Hai?

### Problem
```
study_hours : 1 – 9
attendance  : 55 – 100
salary      : 10,000 – 500,000  ← ye sab dominate karti hai!
```
Distance-based algorithms (KNN, SVM) mein **badi values chhoti values ko dominate** karti hain.

### 2 Types of Scaling

| | StandardScaler | MinMaxScaler |
|--|----------------|--------------|
| **Formula** | (x - mean) / std | (x - min) / (max - min) |
| **Result** | mean=0, std=1 | 0 to 1 |
| **Best for** | KNN, Logistic Regression | Neural Networks |

### Golden Rule 🥇
> `fit_transform()` SIRF training data pe — test pe sirf `transform()`!
> Warna **Data Leakage** hogi!


In [ ]:
print('=== BEFORE SCALING ===')
print(X_train.describe().round(2))

In [ ]:
# StandardScaler
std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)  # fit + transform
X_test_std  = std_scaler.transform(X_test)        # sirf transform!

print('=== AFTER StandardScaler ===')
print(pd.DataFrame(X_train_std, columns=X.columns).describe().round(2))
print()
print('✅ Observe: mean ≈ 0, std ≈ 1 har column mein!')

In [ ]:
# MinMaxScaler
mm_scaler = MinMaxScaler()
X_train_mm = mm_scaler.fit_transform(X_train)
X_test_mm  = mm_scaler.transform(X_test)

print('=== AFTER MinMaxScaler ===')
print(pd.DataFrame(X_train_mm, columns=X.columns).describe().round(2))
print()
print('✅ Observe: min=0, max=1 har column mein!')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].boxplot(X_train.values, labels=X.columns, patch_artist=True)
axes[0].set_title('Before Scaling', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

axes[1].boxplot(X_train_std, labels=X.columns, patch_artist=True)
axes[1].set_title('After StandardScaler\n(mean=0, std=1)', fontweight='bold')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].tick_params(axis='x', rotation=30)

axes[2].boxplot(X_train_mm, labels=X.columns, patch_artist=True)
axes[2].set_title('After MinMaxScaler\n(0 to 1)', fontweight='bold')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Feature Scaling — Before vs After', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## ✅ Feature Scaling — Key Takeaways

| Cheez | Matlab |
|-------|--------|
| **Kyun zaroori** | Distance-based algorithms ke liye |
| **StandardScaler** | Mean=0, Std=1 — most common |
| **MinMaxScaler** | 0 to 1 range |
| **Golden Rule** | `fit_transform()` sirf training pe! |
| **Tree models** | Decision Tree, Random Forest ko scaling ki zaroorat NAHI |


---
---
# 👥 PART 3 — K-NEAREST NEIGHBORS (KNN)
---
---

## 📖 Theory — KNN Kya Hai?

### Simple Idea
> "Batao tumhare dost kaun hain — bata dunga tum kya ho!"

Naye point ke **k sabse paas ke neighbors** dekho — jo class majority mein ho wahi predict karo.

### K Ka Matlab
```
k=1  → 1 neighbor dekho   → overfit ho sakta hai
k=5  → 5 neighbors dekho  → balanced (good default)
k=20 → 20 neighbors dekho → underfit ho sakta hai
```

### ⚠️ Scaling COMPULSORY Hai!
KNN distance calculate karta hai — bina scaling ke badi features dominate karti hain.


In [ ]:
# KNN — Scaling ke saath (ZAROORI!)
std_scaler_knn = StandardScaler()
X_train_knn = std_scaler_knn.fit_transform(X_train)
X_test_knn  = std_scaler_knn.transform(X_test)

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_knn, y_train)

y_pred_knn = knn_model.predict(X_test_knn)
acc_knn = accuracy_score(y_test, y_pred_knn)

print(f'KNN Accuracy (k=5): {acc_knn*100:.1f}%')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_knn))

In [ ]:
# Best K dhundho
k_values = range(1, 31)
test_accs_knn = []
train_accs_knn = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_knn, y_train)
    train_accs_knn.append(accuracy_score(y_train, knn.predict(X_train_knn)))
    test_accs_knn.append(accuracy_score(y_test,  knn.predict(X_test_knn)))

best_k = k_values[np.argmax(test_accs_knn)]

plt.figure(figsize=(12, 5))
plt.plot(k_values, [a*100 for a in train_accs_knn], 'b-o', label='Training', lw=2, ms=5)
plt.plot(k_values, [a*100 for a in test_accs_knn],  'r-o', label='Testing',  lw=2, ms=5)
plt.axvline(x=best_k, color='green', linestyle='--', lw=2, label=f'Best k={best_k}')
plt.xlabel('K (neighbors)'); plt.ylabel('Accuracy %')
plt.title('KNN — Best K Dhundho', fontweight='bold')
plt.legend(); plt.xticks(k_values); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Best k = {best_k}  →  Test Accuracy = {max(test_accs_knn)*100:.1f}%')

In [ ]:
# Scaling ka fark
knn_no_scale = KNeighborsClassifier(n_neighbors=best_k)
knn_no_scale.fit(X_train, y_train)
acc_no_scale = accuracy_score(y_test, knn_no_scale.predict(X_test))

best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train_knn, y_train)
acc_scaled = accuracy_score(y_test, best_knn.predict(X_test_knn))

print('=== SCALING KA FARK ===')
print(f'KNN without scaling : {acc_no_scale*100:.1f}%')
print(f'KNN with scaling    : {acc_scaled*100:.1f}%')
print(f'Improvement         : +{(acc_scaled - acc_no_scale)*100:.1f}%')

## ✅ KNN — Key Takeaways

| Cheez | Matlab |
|-------|--------|
| **Idea** | k nearest neighbors ka vote |
| **k chota** | Complex, overfit ho sakta hai |
| **k bada** | Simple, underfit ho sakta hai |
| **Scaling** | COMPULSORY — bina scaling ke kharab kaam karta hai |
| **Slow predict** | Training data pura scan karta hai |


---
---
# 🔄 PART 4 — CROSS VALIDATION
---
---

## 📖 Theory — Cross Validation Kyun?

### Problem with Simple Train/Test Split
> Ek baar ka test set "lucky" ho sakta hai — model actually weak hai lekin score acha aaya!

### Solution: 5-Fold Cross Validation
```
Fold 1: [TEST ] [train] [train] [train] [train] → Score: 85%
Fold 2: [train] [TEST ] [train] [train] [train] → Score: 83%
Fold 3: [train] [train] [TEST ] [train] [train] → Score: 87%
Fold 4: [train] [train] [train] [TEST ] [train] → Score: 84%
Fold 5: [train] [train] [train] [train] [TEST ] → Score: 86%

Final Score = Average = 85% ± 1.4%  → Reliable!
```


In [ ]:
# Cross Validation — 5-Fold
rf_cv = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

cv_scores = cross_val_score(
    rf_cv, X, y,
    cv=5,               # 5 folds
    scoring='accuracy'
)

print('=== 5-Fold Cross Validation ===')
print()
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score*100:.1f}%')
print()
print(f'Average Accuracy : {cv_scores.mean()*100:.1f}%')
print(f'Std Deviation    : ±{cv_scores.std()*100:.1f}%')
print()
print('Std dev kam = model zyada consistent!')

In [ ]:
# Simple split vs CV comparison
rf_simple = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_simple.fit(X_train, y_train)
simple_acc = accuracy_score(y_test, rf_simple.predict(X_test))

print(f'Simple Train/Test : {simple_acc*100:.1f}%')
print(f'5-Fold CV Average : {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%')

plt.figure(figsize=(10, 5))
plt.bar(range(1, 6), cv_scores*100, color='steelblue', edgecolor='white', width=0.6)
plt.axhline(y=cv_scores.mean()*100, color='red', linestyle='--', lw=2,
            label=f'CV Average: {cv_scores.mean()*100:.1f}%')
plt.axhline(y=simple_acc*100, color='green', linestyle='--', lw=2,
            label=f'Simple Split: {simple_acc*100:.1f}%')
plt.xlabel('Fold'); plt.ylabel('Accuracy %')
plt.title('5-Fold Cross Validation — Scores', fontweight='bold')
plt.legend(); plt.ylim(60, 100)
plt.tight_layout(); plt.show()

## ✅ Cross Validation — Key Takeaways

| Cheez | Matlab |
|-------|--------|
| **Kyun use karein** | Simple split unreliable ho sakta hai |
| **cv=5** | 5-fold — standard choice |
| **cv=10** | Zyada accurate lekin slow |
| **mean()** | Final performance estimate |
| **std()** | Consistency — kam = better |


---
---
# 🎛️ PART 5 — HYPERPARAMETER TUNING
---
---

## 📖 Theory — Hyperparameter Kya Hote Hain?

```
Parameters    (model khud seekhta hai training mein):
  → Regression ke coefficients
  → Tree ke split conditions

Hyperparameters (tum set karte ho pehle se):
  → n_estimators=100
  → max_depth=5
  → n_neighbors=5
```

### Solution: GridSearchCV
> Tumhe sirf possible values deni hain — GridSearchCV har combination try karta hai + CV karta hai + best batata hai!


In [ ]:
# GridSearchCV
param_grid = {
    'n_estimators':      [50, 100, 200],
    'max_depth':         [3, 5, 7],
    'min_samples_split': [5, 10, 20]
}

total = 3 * 3 * 3 * 5
print(f'Total model fits: 3×3×3 combinations × 5 CV folds = {total}')
print('(GridSearchCV ye sab automatically karega!)')
print()

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

In [ ]:
print('=== RESULTS ===')
print(f'Best Parameters : {grid_search.best_params_}')
print(f'Best CV Score   : {grid_search.best_score_*100:.1f}%')
print()

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
print(f'Test Accuracy (Best Params): {accuracy_score(y_test, y_pred_best)*100:.1f}%')

In [ ]:
# Top combinations
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('mean_test_score', ascending=False)

top10 = results_df[['param_n_estimators', 'param_max_depth',
                     'param_min_samples_split', 'mean_test_score']].head(10).copy()
top10['mean_test_score'] = (top10['mean_test_score'] * 100).round(1)
top10.columns = ['n_estimators', 'max_depth', 'min_samples_split', 'CV Score %']
print('Top 10 Combinations:')
print(top10.to_string(index=False))

## ✅ Hyperparameter Tuning — Key Takeaways

| Cheez | Matlab |
|-------|--------|
| **Hyperparameters** | Tum set karte ho — model nahi seekhta |
| **GridSearchCV** | Har combination try karta hai automatically |
| **param_grid** | Dictionary — kaunse values try karein |
| **best_params_** | Best parameter combination |
| **best_estimator_** | Already trained best model |
| **Tip** | `RandomizedSearchCV` = fast alternative (random combinations try karta hai) |


---
---
# 🔵 PART 6 — K-MEANS CLUSTERING (Unsupervised!)
---
---

## 📖 Theory — K-Means Kya Hai?

### Supervised vs Unsupervised
```
Supervised (abhi tak sab):
  Data: [features] → [TARGET]   ← target tha
  Goal: target predict karna

Unsupervised (K-Means):
  Data: [features]              ← koi target NAHI!
  Goal: khud groups (clusters) dhundha
```

### K-Means Steps
```
Step 1: k random centers choose karo
Step 2: Har point nearest center ko assign karo
Step 3: Centers update karo (assigned points ka average)
Step 4: Repeat until stable

Result: k natural groups/clusters!
```

### Real Life Uses
- Customer segmentation (Premium / Regular / Budget)
- Document clustering
- Image compression


In [ ]:
# K-Means — 2 features use karte hain (visualization easy hogi)
X_km = df[['study_hours', 'attendance']].values

scaler_km = StandardScaler()
X_km_scaled = scaler_km.fit_transform(X_km)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_km_scaled)

labels  = kmeans.labels_
centers = kmeans.cluster_centers_

print(f'Clusters found: {len(np.unique(labels))}')
print()
print('Cluster sizes:')
vals, cnts = np.unique(labels, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'  Cluster {v}: {c} students')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_km = ['steelblue', 'tomato', 'green']
for i in range(3):
    mask = labels == i
    axes[0].scatter(X_km[mask, 0], X_km[mask, 1],
                    c=colors_km[i], alpha=0.5, s=30, label=f'Cluster {i}')

centers_orig = scaler_km.inverse_transform(centers)
axes[0].scatter(centers_orig[:, 0], centers_orig[:, 1],
                c='black', marker='*', s=300, zorder=5, label='Centers')
axes[0].set_xlabel('Study Hours'); axes[0].set_ylabel('Attendance %')
axes[0].set_title('K-Means Clustering (k=3)', fontweight='bold')
axes[0].legend()

df_km = df[['study_hours', 'attendance', 'marks']].copy()
df_km['cluster'] = labels
cluster_profile = df_km.groupby('cluster').mean().round(1)
cluster_profile.plot(kind='bar', ax=axes[1], edgecolor='white')
axes[1].set_title('Cluster Profiles (Average values)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout(); plt.show()
print('Cluster Profiles:')
print(cluster_profile)

In [ ]:
# Elbow Method — Best k dhundho
inertias = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_km_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(k_range, inertias, 'bo-', lw=2, ms=8)
plt.axvline(x=3, color='red', linestyle='--', lw=2, label='Elbow at k=3')
plt.xlabel('Number of Clusters (k)'); plt.ylabel('Inertia')
plt.title('Elbow Method — Best k Dhundho', fontweight='bold')
plt.legend(); plt.xticks(k_range); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('Jahan curve mein "elbow" bane = best k!')

## ✅ K-Means — Key Takeaways

| Cheez | Matlab |
|-------|--------|
| **Unsupervised** | Koi target nahi — model khud groups banata hai |
| **k** | Kitne clusters — tum decide karte ho |
| **Elbow Method** | Best k dhundne ka tarika |
| **Inertia** | Within-cluster distance — kam = better clusters |
| **Scaling** | K-Means ke liye bhi zaroori! |
| **n_init=10** | 10 baar random start — best result choose karta hai |


---
# 🏆 FINAL — Sab Algorithms Ka Comparison

In [ ]:
print('=' * 65)
print('      MACHINE LEARNING — COMPLETE ALGORITHMS SUMMARY')
print('=' * 65)
print()
print(f"{'Algorithm':<22} {'Type':<16} {'Scaling?':<10} Metric")
print('-' * 65)
algos = [
    ('Linear Regression',   'Regression',    'No',    'R² Score'),
    ('Logistic Regression', 'Binary Clf',    'Optional', 'Accuracy'),
    ('Decision Tree',       'Multi-class',   'No',    'Accuracy'),
    ('Random Forest',       'Multi-class',   'No',    'Accuracy'),
    ('KNN',                 'Multi-class',   'YES!',  'Accuracy'),
    ('K-Means',             'Clustering',    'YES!',  'Inertia'),
]
for alg, typ, sc, metric in algos:
    print(f'{alg:<22} {typ:<16} {sc:<10} {metric}')

print('=' * 65)
print()
print('Kab Kaunsa Use Karein?')
print()
print('  Numbers predict karna       → Linear Regression')
print('  Pass/Fail, Yes/No           → Logistic Regression')
print('  Multiple categories         → Decision Tree')
print('  Zyada accuracy chahiye      → Random Forest ← almost always best!')
print('  Simple, interpretable       → KNN (scale karna mat bhoolna)')
print('  Groups dhundna (no target)  → K-Means')

---
# 🗺️ Aage Kya Seekhna Hai? (Part 3!)

```
✅ Linear Regression
✅ Logistic Regression
✅ Decision Tree
✅ Random Forest
✅ Feature Scaling
✅ KNN
✅ Cross Validation
✅ Hyperparameter Tuning
✅ K-Means Clustering

⏭️  SVM (Support Vector Machine)  → Powerful boundary — complex data ke liye
⏭️  Neural Networks (basics)      → Deep Learning ki taraf pehla qadam
⏭️  Pipeline                      → Scaling + Model ek saath wrap karo
⏭️  Imbalanced Data               → Jab classes unequal hon (SMOTE, class_weight)
⏭️  PCA                           → Dimensions reduce karna
⏭️  Real Dataset Practice         → Kaggle competitions shuru karo!
```

---

## Complete ML Pipeline — Hamesha Ye Order Follow Karo

```python
# 1. Data Load
df = pd.read_csv('data.csv')

# 2. EDA
df.info(), df.describe(), df.isnull().sum()

# 3. Cleaning

# 4. Feature Engineering

# 5. X aur y
X = df.drop('target', axis=1)
y = df['target']

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

# 7. Scaling (KNN, Logistic ke liye)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# 8. Model
model = RandomForestClassifier()
model.fit(X_train_sc, y_train)

# 9. Cross Validate
scores = cross_val_score(model, X, y, cv=5)
print(f'CV Score: {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%')

# 10. Hyperparameter Tune
grid = GridSearchCV(model, param_grid, cv=5, n_jobs=-1)
grid.fit(X_train_sc, y_train)

# 11. Final Evaluate
y_pred = grid.best_estimator_.predict(X_test_sc)
print(f'Test Accuracy: {accuracy_score(y_test, y_pred)*100:.1f}%')
```
